# Phase 1 — Build the MATCH-only manifest

Parses the EmoMV annotation CSVs, keeps `match_lbl==1` rows only, drops missing/corrupt files, hashes every clip for exact-duplicate removal, flags true duration outliers, windows the long tail, and writes `data/manifest.csv` — the single source of truth every later phase reads from.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # notebooks/ -> project/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from project.config import load_config, resolve_path

cfg = load_config()


In [2]:
import pandas as pd
from project.preprocessing.build_manifest import build_manifest

manifest = build_manifest(cfg)
manifest.head()

PHASE 1 — Building manifest
  DS1/train: 4110 rows -> 2208 MATCH rows (DS1_TRAIN_MATCH_MISMATCH_labels.csv)
  DS1/val: 556 rows -> 310 MATCH rows (DS1_VAL_MATCH_MISMATCH_labels.csv)
  DS1/test: 248 rows -> 124 MATCH rows (DS1_TEST_MATCH_MISMATCH_labels.csv)
  DS2/train: 496 rows -> 248 MATCH rows (DS2_TRAIN_MATCH_MISMATCH_labels.csv)
  DS2/val: 120 rows -> 60 MATCH rows (DS2_VAL_MATCH_MISMATCH_labels.csv)
  DS3/train: 360 rows -> 180 MATCH rows (DS3_TRAIN_MATCH_MISMATCH_labels.csv)
  DS3/val: 96 rows -> 48 MATCH rows (DS3_VAL_MATCH_MISMATCH_labels.csv)
Total MATCH annotation rows parsed: 3178


Hashing videos (SHA-256): 100%|██████████| 3178/3178 [00:05<00:00, 610.12it/s]


Exact duplicates (identical file hash): 34 found, removed
  -> duplicate report: /Users/rafiabhista/Documents/code/apple/c1_audio/emomv_2/project/data/duplicates_report.csv
  [WARN] 2 duplicate pair(s) have CONFLICTING emotion labels (identical video content, different label) -- resolved by keeping the first-seen annotation. Review these rows in duplicates_report.csv manually:
      kept='exciting' (exciting_REVIEW_EXT_Q2(102).mp4)  vs  dropped='tense' (tense_HAPPY_HEA(200)-7-of-8.mp4)
      kept='fear' (fear_TENS_HE(81)-6-of-8.mp4)  vs  dropped='tense' (tense_TENS_LLT(10)).mp4)
Duration outliers (> 35s): 20 clips -> inspect at /Users/rafiabhista/Documents/code/apple/c1_audio/emomv_2/project/data/duration_outliers.csv
Long clips windowed: 20 parent videos -> 65 window rows
----------------------------------------------------------------------
Manifest rows       : 3189
Unique source videos : 3144
Per-dataset counts  :
dataset_source
DS1    2653
DS2     308
DS3     228
dtype: int64
Per-

,video_id,video_path,emotion_label,emotion_id,dataset_source,match_type,video_hash,source_group_id,orig_duration_s,orig_frame_count,fps,window_index,parent_video_id,start_frame,end_frame,window_duration_s,split
0,DS1__exciting_ORG_EXT(130),/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,0,DS1,match,5e0a5f9960e8e5c5335c2bd8fa70b223d4cd9709a2a44e...,DS1__exciting_ORG_EXT(130),31.4982,944,29.970,0,DS1__exciting_ORG_EXT(130),0,944,31.4982,
1,DS1__exciting_TEST_MV(5)-2-of-8,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,0,DS1,match,eaa0f279d353668dd668ab2d7591eb2f9d82dbdffbb2a6...,DS1__exciting_TEST_MV(5),27.7360,665,23.976,0,DS1__exciting_TEST_MV(5)-2-of-8,0,665,27.7360,
2,DS1__exciting_DEAP_DBW_EXT(39),/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,0,DS1,match,0f74dfe3913bead4cfb5e04fe5e0aafdfdeff291b9c94e...,DS1__exciting_DEAP_DBW_EXT(39),29.9600,749,25.000,0,DS1__exciting_DEAP_DBW_EXT(39),0,749,29.9600,
3,DS1__exciting_TEST_MV(6)-3-of-26,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,0,DS1,match,10cad658ba962d5b3390b6fa67d584dba45ab0c1ec78de...,DS1__exciting_TEST_MV(6),29.9966,899,29.970,0,DS1__exciting_TEST_MV(6)-3-of-26,0,899,29.9966,
4,DS1__exciting_ORG_EXT(166),/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,0,DS1,match,5f272bfbeeada30fd531ff7870d913576433c6d851929b...,DS1__exciting_ORG_EXT(166),29.5200,738,25.000,0,DS1__exciting_ORG_EXT(166),0,738,29.5200,


### Duplicate report
Exact (byte-identical) duplicates found via SHA-256 hash. Any row with `label_conflict=True` is the same physical video annotated under two different emotions — a real annotation-noise finding, worth a manual look.

In [3]:
dups = pd.read_csv(resolve_path(cfg, 'paths.duplicates_csv'))
print(f"{len(dups)} exact duplicate pairs removed "
      f"({dups['label_conflict'].sum() if 'label_conflict' in dups else 0} with conflicting labels)")
dups.head(10)

34 exact duplicate pairs removed (2 with conflicting labels)


,video_hash,kept_path,kept_label,dropped_path,dropped_label,label_conflict
0,42eda1068161c85c5d89d0966db5794301f6c6d6ef8093...,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,False
1,4eb2b5d63da82bfc433a262906e2e84305c32eacf354c2...,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,False
2,7a9e93d90da8431a894aed0b6a37c231f23880ff171dde...,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,False
3,ec5b0a8a2d4dce73f3e0137a8e6affd03c070c3e254d8d...,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,False
4,1abd8a54ec02ff0364fc9767b360c858fbcb2ef6f8409c...,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,False
5,fb63a30a6fd8df978eac1da0932869ea84122aee1d2c76...,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,/Users/rafiabhista/Documents/code/apple/c1_aud...,exciting,False
6,8beb8e5415e0fab6a32baa95649b5185428e55a1169a8f...,/Users/rafiabhista/Documents/code/apple/c1_aud...,fear,/Users/rafiabhista/Documents/code/apple/c1_aud...,fear,False
7,430588ed54fc8df913e2ec8d646d7b44ad5fefbfa98d15...,/Users/rafiabhista/Documents/code/apple/c1_aud...,fear,/Users/rafiabhista/Documents/code/apple/c1_aud...,fear,False
8,a301fb063eb9e0853fc727a156edf6d9ca3def51c17621...,/Users/rafiabhista/Documents/code/apple/c1_aud...,fear,/Users/rafiabhista/Documents/code/apple/c1_aud...,fear,False
9,d766429b5a4bc3a32549e528780e6e95921f85edfa4cb5...,/Users/rafiabhista/Documents/code/apple/c1_aud...,tense,/Users/rafiabhista/Documents/code/apple/c1_aud...,tense,False


### Duration outliers
Clips exceeding `max_duration_s + tolerance_s` (i.e. genuinely long, not just encoding-rounding noise around 30s) — these are the ones that get windowed.

In [4]:
outliers = pd.read_csv(resolve_path(cfg, 'paths.duration_outliers_csv'))
print(f"{len(outliers)} outlier source videos")
outliers[['dataset_source','stem','emotion_label','duration']].sort_values('duration', ascending=False)

20 outlier source videos


,dataset_source,stem,emotion_label,duration
19,DS1,sad_FINAL_DL(90),sad,392.366667
5,DS1,tense_TENS_HE(161),tense,296.880000
2,DS1,exciting_ORG_EXT(88),exciting,91.160000
10,DS1,sad_ORG_SAD(135),sad,90.280000
7,DS1,sad_ORG_SAD(293),sad,89.338964
11,DS1,sad_ORG_SAD(278),sad,89.080000
12,DS1,sad_TT(8),sad,88.566667
17,DS1,tense_ORG_TENS(67),tense,87.466667
0,DS1,exciting_ORG_EXT(189),exciting,60.769019
14,DS1,relax_REVIEW_RLX(34),relax,59.993293
